# General Selector Summary

This notebook summarizes selector result folders under one `RESULTS_ROOT`.

It supports naming patterns for:

- combined cost+rank selectors, for example `..._0.9lw_70_30_lw0.9_MAE_RankNet`
- cost-only selectors, for example `..._1.0lw_70_30_costonly_MAE`
- rank-only selectors if you later use names like `..._rankonly_ListNet`

For every selector, the selected solver is `argmin(pred_costs)`. The reported cost/rank are the corresponding true `label_costs` and tie-aware ranks recomputed from `label_costs`. VBS, SBS, and the five fixed solvers are computed from the true labels on the same test set, so the aggregate table reports mean/median/std cost and rank for every method family.

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

RESULTS_ROOT = Path('60s_v2')  # change to Path('10s_v2'), Path('60s_v2'), Path('2s'), etc.
PREDICTION_FILE = 'predictions_epoch30.json'
SOLVERS = ['CLK', 'EAX', 'LKH', 'MAOS', 'CONCORDE']

pd.set_option('display.max_rows', 80)
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)

In [ ]:
def parse_selector_metadata(folder_name: str) -> dict:
    base_match = re.search(
        r'results_(?P<cutoff>[0-9.]+s)_(?P<hidden>\d+)hd_(?P<lr>[^_]+)lr_(?P<dr>[^_]+)dr_'
        r'(?P<cost_loss>MSE|MAE|Huber)_(?P<rank_loss>RankNet|ListNet|LambdaRank)_'
        r'(?P<rank_method>[^_]+)_(?P<loss_weight>[0-9.]+)lw(?P<suffix>.*)$',
        folder_name,
    )
    if not base_match:
        return {
            'method': folder_name,
            'objective_type': 'unknown',
            'cost_loss': None,
            'rank_loss': None,
            'loss_weight': np.nan,
            'folder': folder_name,
        }

    meta = base_match.groupdict()
    suffix = meta['suffix']
    loss_weight = float(meta['loss_weight'])
    cost_loss = meta['cost_loss']
    rank_loss = meta['rank_loss']

    cost_only_match = re.search(r'costonly_(MSE|MAE|Huber)$', suffix)
    rank_only_match = re.search(r'rankonly_(RankNet|ListNet|LambdaRank)$', suffix)
    combined_match = re.search(r'lw[0-9.]+_(MSE|MAE|Huber)_(RankNet|ListNet|LambdaRank)$', suffix)

    if 'skewed700300' in suffix:
        split_type = 'skewed700300'
    elif 'stratified' in suffix:
        split_type = 'stratified'
    elif '70_30' in suffix:
        split_type = '70_30'
    else:
        split_type = 'unknown'

    if cost_only_match or np.isclose(loss_weight, 1.0):
        objective_type = 'cost_only'
        shown_cost_loss = cost_only_match.group(1) if cost_only_match else cost_loss
        method = f'{shown_cost_loss} cost-only'
    elif rank_only_match or np.isclose(loss_weight, 0.0):
        objective_type = 'rank_only'
        shown_rank_loss = rank_only_match.group(1) if rank_only_match else rank_loss
        method = f'{shown_rank_loss} rank-only'
    elif combined_match:
        objective_type = 'cost_rank'
        method = f'{combined_match.group(1)}+{combined_match.group(2)} lw={loss_weight:g}'
    else:
        objective_type = 'cost_rank'
        method = f'{cost_loss}+{rank_loss} lw={loss_weight:g}'

    return {
        'method': method,
        'objective_type': objective_type,
        'cost_loss': cost_loss,
        'rank_loss': rank_loss,
        'loss_weight': loss_weight,
        'split_type': split_type,
        'folder': folder_name,
    }



def average_tie_ranks(costs, tolerance=1e-12):
    """Return ascending average ranks from costs using standard tie handling.

    Example: if three solvers tie for the smallest cost, they occupy
    positions 1, 2, and 3, so each receives rank 2. The next distinct
    cost receives rank 4, then rank 5, etc. The rank sum is always
    1 + 2 + ... + n.
    """
    costs = np.asarray(costs, dtype=float)
    order = np.argsort(costs, kind='mergesort')
    ranks = np.empty(len(costs), dtype=float)
    position = 1
    i = 0
    while i < len(order):
        j = i + 1
        while j < len(order) and abs(costs[order[j]] - costs[order[i]]) <= tolerance:
            j += 1
        tied_positions = np.arange(position, position + (j - i), dtype=float)
        ranks[order[i:j]] = tied_positions.mean()
        position += j - i
        i = j
    return ranks

def load_predictions(path: Path) -> pd.DataFrame:
    metadata = parse_selector_metadata(path.parent.name)
    with path.open() as handle:
        records = json.load(handle)

    rows = []
    for record in records:
        pred_costs = np.asarray(record['pred_costs'], dtype=float)
        label_costs = np.asarray(record['label_costs'], dtype=float)
        label_ranks = average_tie_ranks(label_costs)
        selected_index = int(np.argmin(pred_costs))
        rows.append({
            **metadata,
            'instance_id': record['instance_id'],
            'selected_solver': SOLVERS[selected_index],
            'selected_solver_index': selected_index,
            'selected_pred_cost': pred_costs[selected_index],
            'selected_cost': label_costs[selected_index],
            'selected_rank': label_ranks[selected_index],
            'label_costs': label_costs,
            'label_ranks': label_ranks,
        })
    return pd.DataFrame(rows)

In [ ]:
prediction_paths = sorted(RESULTS_ROOT.glob(f'*/{PREDICTION_FILE}'))
print(f'Found {len(prediction_paths)} prediction files')
for path in prediction_paths:
    meta = parse_selector_metadata(path.parent.name)
    print(f"{meta['method']:<28} {meta['objective_type']:<10} {path.parent.name}")

selector_df = pd.concat([load_predictions(path) for path in prediction_paths], ignore_index=True)

instances_by_method = selector_df.groupby('method')['instance_id'].apply(set)
same_test_set = all(instances == instances_by_method.iloc[0] for instances in instances_by_method)
print(f'Same test instances across methods: {same_test_set}')
print(f'Test instances per method: {len(instances_by_method.iloc[0])}')

In [ ]:
def summarize(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby(['method', 'objective_type', 'cost_loss', 'rank_loss', 'loss_weight'], dropna=False)
        .agg(
            n_instances=('instance_id', 'count'),
            cost_mean=('cost', 'mean'),
            cost_median=('cost', 'median'),
            cost_std=('cost', 'std'),
            rank_mean=('rank', 'mean'),
            rank_median=('rank', 'median'),
            rank_std=('rank', 'std'),
        )
        .reset_index()
        .sort_values(['cost_mean', 'rank_mean'], kind='mergesort')
    )


selector_metrics = selector_df.rename(columns={
    'selected_cost': 'cost',
    'selected_rank': 'rank',
})[['method', 'objective_type', 'cost_loss', 'rank_loss', 'loss_weight', 'split_type', 'folder', 'instance_id', 'selected_solver', 'cost', 'rank']]

selector_summary = summarize(selector_metrics)
display(selector_summary)

In [ ]:
# True labels, VBS, and SBS.
label_df = (
    selector_df.sort_values(['method', 'instance_id'])
    .drop_duplicates('instance_id')
    [['instance_id', 'label_costs', 'label_ranks']]
    .copy()
)

solver_rows = []
for _, row in label_df.iterrows():
    for solver_index, solver in enumerate(SOLVERS):
        solver_rows.append({
            'instance_id': row['instance_id'],
            'solver': solver,
            'cost': row['label_costs'][solver_index],
            'rank': row['label_ranks'][solver_index],
        })

solver_df = pd.DataFrame(solver_rows)
solver_metrics = (
    solver_df
    .assign(
        method=solver_df['solver'],
        objective_type='fixed_solver',
        cost_loss=None,
        rank_loss=None,
        loss_weight=np.nan,
        split_type='baseline',
        folder=None,
        selected_solver=solver_df['solver'],
    )
    [['method', 'objective_type', 'cost_loss', 'rank_loss', 'loss_weight', 'split_type', 'folder', 'instance_id', 'selected_solver', 'cost', 'rank']]
)
solver_summary = summarize(solver_metrics)
display(solver_summary)

sbs_solver = solver_df.groupby('solver')['cost'].mean().sort_values().index[0]
print(f'SBS solver selected by lowest mean true cost: {sbs_solver}')

vbs_rows = []
for _, row in label_df.iterrows():
    costs = np.asarray(row['label_costs'], dtype=float)
    ranks = np.asarray(row['label_ranks'], dtype=float)
    best_index = int(np.argmin(costs))
    vbs_rows.append({
        'method': 'VBS',
        'objective_type': 'oracle',
        'cost_loss': None,
        'rank_loss': None,
        'loss_weight': np.nan,
        'split_type': 'baseline',
        'folder': None,
        'instance_id': row['instance_id'],
        'selected_solver': SOLVERS[best_index],
        'cost': costs[best_index],
        'rank': ranks[best_index],
    })

vbs_metrics = pd.DataFrame(vbs_rows)
sbs_metrics = (
    solver_df[solver_df['solver'] == sbs_solver]
    .assign(
        method=f'SBS ({sbs_solver})',
        objective_type='single_best',
        cost_loss=None,
        rank_loss=None,
        loss_weight=np.nan,
        split_type='baseline',
        folder=None,
        selected_solver=sbs_solver,
    )
    [['method', 'objective_type', 'cost_loss', 'rank_loss', 'loss_weight', 'split_type', 'folder', 'instance_id', 'selected_solver', 'cost', 'rank']]
)

all_metrics = pd.concat([selector_metrics, vbs_metrics, sbs_metrics, solver_metrics], ignore_index=True)
all_summary = summarize(all_metrics)

priority = {'VBS': 0, f'SBS ({sbs_solver})': 1, **{solver: 2 for solver in SOLVERS}}
all_summary['display_order'] = all_summary['method'].map(priority).fillna(3)
all_summary = all_summary.sort_values(['display_order', 'cost_mean', 'rank_mean']).drop(columns='display_order')
display(all_summary)

In [ ]:
cost_only_summary = all_summary[
    all_summary['objective_type'].isin(['cost_only', 'oracle', 'single_best', 'fixed_solver'])
].copy()
display(cost_only_summary)

In [ ]:
selection_counts = (
    all_metrics.groupby(['method', 'selected_solver'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=SOLVERS, fill_value=0)
)
display(selection_counts)

In [ ]:
output_dir = RESULTS_ROOT / 'summary_tables_general'
output_dir.mkdir(exist_ok=True)

all_summary.to_csv(output_dir / 'all_selectors_vbs_sbs_summary.csv', index=False)
selector_summary.to_csv(output_dir / 'selectors_only_summary.csv', index=False)
solver_summary.to_csv(output_dir / 'fixed_solver_summary.csv', index=False)
cost_only_summary.to_csv(output_dir / 'cost_only_vbs_sbs_summary.csv', index=False)
all_metrics.to_csv(output_dir / 'per_instance_all_selectors_vbs_sbs_metrics.csv', index=False)
selection_counts.to_csv(output_dir / 'selection_counts.csv')

print(f'Wrote summary tables to: {output_dir.resolve()}')